# NeuralAI-Air-135M — SFT Training (v19)

Supervised Fine-Tuning for the custom 135M-parameter Llama-architecture model.

**Model:** NeuralAI-Air-135M (15 layers, 768 hidden, 32000 vocab, GQA 12/2 heads)
**Data:** 1000+ ChatML instruction examples
**Hardware:** T4 GPU (free Colab) or better
**Output:** Fine-tuned model + tokenizer for GGUF conversion

## Setup

1. Upload `NeuralAI-Air-135M-HF/` directory to your Google Drive root
2. Upload `train_sft_v19.jsonl` to your Google Drive root
3. Run all cells in order

In [ ]:
# Install dependencies (Colab has most pre-installed)
!pip install -q transformers datasets accelerate safetensors

In [ ]:
import os, json, torch
from pathlib import Path
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)
from google.colab import drive

print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
assert torch.cuda.is_available(), "Enable GPU in Runtime → Change runtime type"

In [ ]:
# Mount Google Drive
drive.mount('/content/drive')

In [ ]:
# Copy model and data from Drive to Colab local (faster I/O)
import shutil, os
from pathlib import Path

# Try multiple possible Drive paths
drive_paths = [
    '/content/drive/MyDrive',
    '/content/drive/My Drive',
]
drive_root = None
for p in drive_paths:
    if Path(p).exists():
        drive_root = p
        break
assert drive_root, 'Google Drive not mounted. Run the mount cell above first.'
print(f'Using Drive root: {drive_root}')

# Copy model dir
src_model = Path(drive_root) / 'NeuralAI-Air-135M-HF'
if not src_model.exists():
    # Try alternate locations
    alt = Path(drive_root) / 'colab_upload_v19' / 'NeuralAI-Air-135M-HF'
    if alt.exists():
        src_model = alt
    else:
        raise FileNotFoundError(f'NeuralAI-Air-135M-HF not found in Drive. Upload it first. Tried: {src_model}')
shutil.copytree(src_model, '/content/model', dirs_exist_ok=True)

# Copy data files
for fname in ['train_sft_v19.jsonl', 'train_dpo_v19.jsonl']:
    src = Path(drive_root) / fname
    if not src.exists():
        alt = Path(drive_root) / 'colab_upload_v19' / fname
        if alt.exists():
            src = alt
    if src.exists():
        shutil.copy(src, f'/content/{fname}')
        print(f'Copied {fname}')
    else:
        print(f'WARNING: {fname} not found in Drive')

# Verify critical files
assert Path('/content/model/config.json').exists(), 'config.json missing!'
assert Path('/content/model/tokenizer.json').exists(), 'tokenizer.json missing!'

# Check for model weights
has_weights = Path('/content/model/model.safetensors').exists()
if has_weights:
    size_mb = Path('/content/model/model.safetensors').stat().st_size / 1e6
    print(f'   model.safetensors size: {size_mb:.1f} MB')
    if size_mb < 500:
        print('   WARNING: file seems truncated! Expected ~511 MB')
        has_weights = False
if not has_weights:
    print('\n' + '='*60)
    print('WARNING: model.safetensors NOT FOUND')
    print('='*60)
    print('The 135M base weights are missing. Training will start')
    print('from RANDOM initialization. With only ~200K tokens, this')
    print('will NOT produce a coherent model.')
    print('\nTo fix, upload model.safetensors from ZO to:')
    print('  /content/model/model.safetensors')
    print('or transfer from ZO host:')
    print('  scp /home/workspace/Projects/NeuralAI/NeuralAI-Air-135M-HF/model.safetensors \\
    print('      your-colab-env:/content/model/')
    print('='*60)
else:
    print('✅ model.safetensors found')

print('\nAll critical files verified')
!ls -lh /content/model/
!wc -l /content/train_sft_v19.jsonl


In [ ]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained('/content/model', trust_remote_code=False)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Tokenizer vocab: {tokenizer.vocab_size}")
print(f"BOS: {tokenizer.bos_token_id}, EOS: {tokenizer.eos_token_id}, PAD: {tokenizer.pad_token_id}")

In [ ]:
# Load model (fp16 for VRAM efficiency)
model = AutoModelForCausalLM.from_pretrained(
    '/content/model',
    torch_dtype=torch.float16,
    trust_remote_code=False,
).cuda()

total_params = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"{total_params/1e6:.1f}M total params, {trainable/1e6:.1f}M trainable")

In [ ]:
# Load and tokenize dataset
MAX_LENGTH = 1024

with open('/content/data.jsonl') as f:
    records = [json.loads(line) for line in f if line.strip()]

def tokenize(examples):
    return tokenizer(
        examples['text'],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )

dataset = Dataset.from_list(records)
dataset = dataset.map(tokenize, batched=True, remove_columns=['text'])

# 90/10 train/eval split
split = dataset.train_test_split(test_size=0.1, seed=42)
train_ds, eval_ds = split['train'], split['test']
print(f"Train: {len(train_ds)}, Eval: {len(eval_ds)}")

In [ ]:
# Training configuration
training_args = TrainingArguments(
    output_dir='/content/checkpoints',
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,        # effective batch = 16
    learning_rate=2e-5,
    num_train_epochs=3,
    warmup_ratio=0.03,
    weight_decay=0.01,
    logging_steps=10,
    save_steps=200,
    save_total_limit=3,
    eval_strategy='steps',
    eval_steps=200,
    fp16=True,
    report_to='none',
    dataloader_num_workers=2,
    remove_unused_columns=False,
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer, mlm=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=data_collator,
    tokenizer=tokenizer,
)

print(f"Steps per epoch: {len(train_ds) // 16}")
print(f"Total steps: {len(train_ds) // 16 * 3}")

In [ ]:
# Train! (15-30 min on T4 with 1000 examples, 3 epochs)
trainer.train()

In [ ]:
# Save final model
OUTPUT_DIR = '/content/drive/MyDrive/NeuralAI-Air-135M-SFT-v19'
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"Model saved to {OUTPUT_DIR}")
!ls -lh {OUTPUT_DIR}/

In [ ]:
# Quick inference test
model.eval()
test_prompt = "<|im_start|>user\nHello, who are you?\n<|im_end|>\n<|im_start|>assistant\n"
inputs = tokenizer(test_prompt, return_tensors='pt').to('cuda')
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=50,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.pad_token_id,
    )
response = tokenizer.decode(outputs[0], skip_special_tokens=False)
print("MODEL OUTPUT:")
print(response)

## DPO Training (v19)

Direct Preference Optimization on 350 preference pairs.
Run this AFTER SFT completes.

In [ ]:
!pip install -q trl peft
from trl import DPOTrainer
from peft import LoraConfig, get_peft_model

# Load DPO data
!cp /content/drive/MyDrive/train_dpo_v19.jsonl /content/dpo.jsonl
with open('/content/dpo.jsonl') as f:
    dpo_records = [json.loads(line) for line in f if line.strip()]
print(f'DPO pairs: {len(dpo_records)}')

# Tokenize DPO pairs
def tokenize_dpo(rec):
    return {
        'prompt': rec['prompt'],
        'chosen': rec['chosen'],
        'rejected': rec['rejected'],
    }

dpo_ds = Dataset.from_list([tokenize_dpo(r) for r in dpo_records])

# Apply LoRA for DPO
lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# DPO args
dpo_args = TrainingArguments(
    output_dir='/content/dpo_checkpoints',
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=5e-6,
    num_train_epochs=3,
    warmup_ratio=0.1,
    logging_steps=5,
    save_steps=50,
    fp16=True,
    report_to='none',
    remove_unused_columns=False,
)

dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,  # use implicit reference
    args=dpo_args,
    train_dataset=dpo_ds,
    tokenizer=tokenizer,
    beta=0.1,
    max_length=1024,
    max_prompt_length=512,
)

dpo_trainer.train()

# Save DPO adapter
DPO_OUT = '/content/drive/MyDrive/NeuralAI-Air-135M-DPO-v19'
model.save_pretrained(DPO_OUT)
tokenizer.save_pretrained(DPO_OUT)
print(f'DPO adapter saved to {DPO_OUT}')


## Post-Training: GGUF Conversion

After training completes, convert the model to GGUF for llama.cpp inference:

```bash
# On ZO host, run:
python3 convert_air_to_gguf.py \
  --weights /path/to/NeuralAI-Air-135M-SFT-v19/model.safetensors \
  --config /content/model/config.json \
  --tokenizer /content/model/tokenizer.json \
  --out NeuralAI-Air-135M-SFT-v19.F32.gguf \
  --quant f16

# Then update services/neuralai_llama_server.sh to point to new GGUF
# Restart neuralai-web-ui service
```